# 🔁 ReAct Agents

**Reasoning and Acting agents**

---

## 📋 Overview

**What you'll learn:**
- ReAct pattern (Reason + Act)
- Thought-Action-Observation loop
- Self-reflection
- Error recovery
- Implementation from scratch

**Time estimate:** ⏱️ 55 minutes | **Difficulty:** 🔴 Advanced

---

In [ ]:
from openai import OpenAI
import os
import json
from typing import Dict, List
import re

client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))

print("✅ Setup complete")

## 🤔 What is ReAct?

### Traditional Agent:
```
Goal → Action → Result
```

### ReAct Agent:
```
Goal → Thought → Action → Observation → Thought → Action → ...
       ↑_______________↓
       (Reasoning Loop)
```

### ReAct Loop:

**1. Thought** (Reasoning)
```
"I need to find the weather in Paris.
 I should use the get_weather tool."
```

**2. Action** (Acting)
```
get_weather(location="Paris")
```

**3. Observation** (Result)
```
{"temp": 18, "condition": "Sunny"}
```

**4. Thought** (Reflect)
```
"Good! Paris is 18°C and sunny.
 Now I can answer the user."
```

**5. Final Answer**
```
"The weather in Paris is 18°C and sunny!"
```

### Benefits:

- 🧠 **Better reasoning**: Explicit thought process
- 🔄 **Self-correction**: Can reflect and adjust
- 📝 **Interpretable**: See agent's thinking
- 🎯 **More accurate**: Thinks before acting

## 🏗️ ReAct Agent Implementation

In [ ]:
class ReActAgent:
    """ReAct (Reasoning + Acting) Agent."""
    
    def __init__(self, tools: Dict, max_iterations: int = 10):
        self.tools = tools
        self.max_iterations = max_iterations
        self.client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))
    
    def run(self, question: str) -> str:
        """Run ReAct agent."""
        
        print(f"❓ Question: {question}\n")
        print("="*60)
        
        # Build ReAct prompt
        react_prompt = self._build_react_prompt(question)
        
        scratchpad = ""
        
        for iteration in range(self.max_iterations):
            print(f"\n🔁 Iteration {iteration + 1}")
            
            # Get next thought/action
            response = self.client.chat.completions.create(
                model="gpt-4",
                messages=[{
                    "role": "user",
                    "content": react_prompt + scratchpad
                }],
                temperature=0,
                stop=["\nObservation:"]  # Stop at observation
            )
            
            output = response.choices[0].message.content
            scratchpad += output
            
            # Check if done
            if "Final Answer:" in output:
                final_answer = output.split("Final Answer:")[1].strip()
                print(f"\n✅ Final Answer: {final_answer}")
                return final_answer
            
            # Parse thought and action
            thought_match = re.search(r"Thought: (.+)", output)
            action_match = re.search(r"Action: ([\w_]+)\[(.+?)\]", output)
            
            if thought_match:
                thought = thought_match.group(1).strip()
                print(f"💭 Thought: {thought}")
            
            if action_match:
                action_name = action_match.group(1)
                action_input = action_match.group(2)
                
                print(f"🔧 Action: {action_name}[{action_input}]")
                
                # Execute action
                observation = self._execute_action(action_name, action_input)
                print(f"👀 Observation: {observation}")
                
                # Add observation to scratchpad
                scratchpad += f"\nObservation: {observation}\n"
            else:
                # No valid action found
                print("⚠️  No valid action found")
                break
        
        return "Max iterations reached without answer"
    
    def _build_react_prompt(self, question: str) -> str:
        """Build ReAct prompt template."""
        
        tool_descriptions = "\n".join([
            f"- {name}: {info['description']}"
            for name, info in self.tools.items()
        ])
        
        prompt = f"""Answer the following question using this format:

Question: [the question]
Thought: [your reasoning about what to do next]
Action: [tool_name][input]
Observation: [result from tool]
... (repeat Thought/Action/Observation as needed)
Thought: [final reasoning]
Final Answer: [your answer to the question]

Available tools:
{tool_descriptions}

Question: {question}
"""
        return prompt
    
    def _execute_action(self, action_name: str, action_input: str) -> str:
        """Execute a tool action."""
        
        if action_name not in self.tools:
            return f"Error: Tool '{action_name}' not found"
        
        try:
            result = self.tools[action_name]['function'](action_input)
            return str(result)
        except Exception as e:
            return f"Error: {str(e)}"

# Define tools
def search(query: str) -> str:
    """Search the web."""
    # Mock implementation
    results = {
        "paris population": "Paris has a population of approximately 2.2 million",
        "eiffel tower height": "The Eiffel Tower is 330 meters tall",
    }
    
    for key in results:
        if key in query.lower():
            return results[key]
    
    return "No results found"

def calculate(expression: str) -> str:
    """Calculate mathematical expressions."""
    try:
        # Safe eval for demo (use a proper math parser in production!)
        result = eval(expression, {"__builtins__": {}}, {})
        return str(result)
    except:
        return "Error: Invalid expression"

tools = {
    "search": {
        "description": "Search for information on the web",
        "function": search
    },
    "calculate": {
        "description": "Perform mathematical calculations",
        "function": calculate
    }
}

# Run agent
agent = ReActAgent(tools=tools)
answer = agent.run("What is the population of Paris divided by 1000?")

print("\n" + "="*60)
print(f"\n✅ Final: {answer}")

## 🔄 Self-Correction Example

In [ ]:
print("🔄 ReAct Self-Correction Example\n")
print("="*60)
print("""
Question: "How tall is the Eiffel Tower in feet?"

Iteration 1:
  Thought: I need to find the height of the Eiffel Tower
  Action: search[eiffel tower height]
  Observation: The Eiffel Tower is 330 meters tall

Iteration 2:
  Thought: I got the height in meters (330m) but the question asks for feet.
           I need to convert meters to feet.
           1 meter = 3.28084 feet
  Action: calculate[330 * 3.28084]
  Observation: 1082.6772

Iteration 3:
  Thought: Now I have the answer. The Eiffel Tower is approximately
           1,083 feet tall.
  Final Answer: The Eiffel Tower is approximately 1,083 feet tall.

💡 The agent:
  1. Realized it got meters instead of feet
  2. Self-corrected by converting units
  3. Provided the correct answer
""")

## ✅ Summary

### ReAct Pattern:

```
Question → Thought → Action → Observation
              ↑                    |
              └────────────────────┘
           (Loop until answer found)
```

### Format:

```
Thought: [reasoning]
Action: [tool_name][input]
Observation: [result]
Thought: [reflect on result]
Action: [next action]
...
Thought: [final reasoning]
Final Answer: [answer]
```

### Benefits of ReAct:

**1. Better Reasoning**
- Explicit thought process
- Step-by-step logic
- More accurate

**2. Self-Correction**
- Can catch mistakes
- Adjust approach
- Try alternatives

**3. Interpretability**
- See agent's thinking
- Debug easily
- Understand failures

**4. Flexibility**
- Handles complex tasks
- Multi-step reasoning
- Dynamic planning

### Implementation Tips:

**1. Clear Format**
```python
# Use strict format in prompt
format = """
Thought: ...
Action: tool_name[input]
Observation: ...
"""
```

**2. Stop Sequences**
```python
# Stop at "Observation:" so you can add it
stop=["\nObservation:"]
```

**3. Parsing**
```python
# Extract thought, action, observation
thought = re.search(r"Thought: (.+)", text)
action = re.search(r"Action: (\w+)\[(.+?)\]", text)
```

**4. Scratchpad**
```python
# Accumulate conversation
scratchpad = ""
scratchpad += thought + action + observation
```

### When to Use ReAct:

✅ **Use ReAct for:**
- Complex reasoning tasks
- Multi-step problems
- When you need interpretability
- Research/exploration

❌ **Use simple agents for:**
- Simple tool calls
- Single-step tasks
- Speed-critical applications

### ReAct vs Simple Agent:

| Aspect | Simple Agent | ReAct |
|--------|-------------|--------|
| **Reasoning** | Implicit | Explicit |
| **Speed** | Faster | Slower |
| **Accuracy** | Good | Better |
| **Interpretability** | Low | High |
| **Self-correction** | No | Yes |

### Next: `07_agents_tools/05_memory.ipynb`